In [1]:
import numpy as np
import pandas as pd
import os
import re
from utils import get_codebook

In [2]:
#directory with the original files from covid impact survey
survey_dir = os.path.join(os.getcwd(),  'associatedpress-covid-impact-survey-public-data')
#directory to store all data that could be used in a phone survey
phone_survey_dir = os.path.join(os.getcwd(),  'phone-survey-data')
if not os.path.exists(phone_survey_dir):
    os.makedirs(phone_survey_dir)

In [12]:
def extract_reponse_id(text):
    if not isinstance(text, str):
        return text
    pattern = r"\((\d+)\)"
    match = re.search(pattern, text)
    if match:
        return int(match.group(1))
    return text

print(extract_reponse_id('(3) Some'))
print(extract_reponse_id('(98) SKIPPED ON WEB'))
print(extract_reponse_id('$75,000 to under $100,000'))

3
98
$75,000 to under $100,000


In [22]:
def extract_clean_response_text(response_text):
    if not isinstance(response_text, str):
        return response_text
    text = response_text.strip()
    if ')' in text:
        text = text[text.index(')')+1:].strip()
    return text
print(extract_clean_response_text('(3) Some'))
print(extract_clean_response_text('(98) SKIPPED ON WEB'))
print(extract_clean_response_text('$75,000 to under $100,000'))

Some
SKIPPED ON WEB
$75,000 to under $100,000


In [13]:
survey_question_codes = ['AGE7',
                         'GENDER',
                         'RACETH',
                         'HHINCOME',
                         'EDUCATION',
                         'HHSIZE1',
                         'HH01S',
                         'HH25S',
                         'HH612S',
                         'HH1317S',
                         'HH18OVS',
                         'SOC2A', 
                         'SOC2B',    
                         'SOC5A', 
                         'SOC5B', 
                         'SOC5C', 
                         'SOC5D', 
                         'SOC5E',
                         'ECON1',
                         'PHYS8',
                         'PHYS4',
                         'PHYS5',
                         'PHYS1B', 
                         'PHYS1C', 
                         'PHYS1D', 
                         'PHYS1E', 
                         'PHYS1F', 
                         'PHYS1G', 
                         'PHYS1H', 
                         'PHYS1I', 
                         'PHYS1J',
                         'PHYS11', 
                         'PHYS11_TEMP'
                        ]
print(f"Total number of questions: {len(survey_question_codes)}")

Total number of questions: 33


In [5]:
filenames = {}
filenames['april'] = '01_April_30_covid_impact_survey.csv'
filenames['may']   = '02_May_12_covid_impact_survey.csv'
filenames['june']  = '03_June_9_covid_impact_survey.csv'

In [6]:
codebook = get_codebook()

In [24]:
for _, filename in filenames.items():
    print(filename)
    file_path = os.path.join(survey_dir,  filename)
    df = pd.read_csv(file_path)
    
    #retrieve and save relevant questions
    survey_df = df[survey_question_codes]
    phone_survey_file_path = os.path.join(phone_survey_dir,  f"full_phone_survey_{filename}")
    survey_df.to_csv(phone_survey_file_path)

    numeric_survey_df = pd.DataFrame()
    #get numeric ID of answers for each question in the survey
    for question_code in survey_df:
        qa = codebook[question_code]
        #retrieve numeric ID of the response
        if question_code == 'HHINCOME':
            numeric_survey_df[question_code]= survey_df[question_code].apply(qa.get_response_id)
        else:
             numeric_survey_df[question_code] = survey_df[question_code].apply(extract_reponse_id)
    #save data file with response ids
    numeric_phone_survey_file_path = os.path.join(phone_survey_dir,  f"numeric_full_phone_survey_{filename}")
    numeric_survey_df.to_csv(numeric_phone_survey_file_path)


    text_survey_df = pd.DataFrame()
    #get numeric ID of answers for each question in the survey
    for question_code in survey_df:
        qa = codebook[question_code]
        #retrieve numeric ID of the response
        #print(question_code)
        text_survey_df[question_code] = survey_df[question_code].apply(extract_clean_response_text)
    #save data file with response ids
    text_phone_survey_file_path = os.path.join(phone_survey_dir,  f"clean_text_full_phone_survey_{filename}")
    text_survey_df.to_csv(text_phone_survey_file_path)


01_April_30_covid_impact_survey.csv


/var/folders/0f/lqjqfrb16p5gf3__wqf2c94h1tfy6x/T/ipykernel_10012/1894401860.py:4: DtypeWarning: Columns (96,156,168,170) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


02_May_12_covid_impact_survey.csv


/var/folders/0f/lqjqfrb16p5gf3__wqf2c94h1tfy6x/T/ipykernel_10012/1894401860.py:4: DtypeWarning: Columns (172) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


03_June_9_covid_impact_survey.csv


In [ ]:
print(survey_df['HHINCOME'])